# **Accès et utilisation des moustiquaires imprégnées d'insecticide**
# Données issues des Enquêtes Démographiques et de Santé (EDS - DHS)

Ce rapport génère des visualisations pour les indicateurs relatifs aux **moustiquaires imprégnées d'insecticide (MII)** à partir des données de l'Enquête Démographique et de Santé (EDS - DHS).

Les deux indicateurs repris concernent **l'accès aux MII** et **l'utilisation des MII**.

---

**L'accès aux MII** représente le pourcentage de la population de fait du ménage, ayant accès à une moustiquaire imprégnée d’insecticide (MII) dans ménage, défini comme la proportion de la population de fait du ménage qui dormirait sous une MII si chaque MII du ménage était utilisée par deux personnes au maximum.

* *Numérateur* : le nombre de personnes de fait qui pourraient dormir sous une MII si chaque MII du ménage était utilisée par deux personnes au maximum, calculé pour chaque ménage comme étant la valeur minimale entre :
    1. le nombre de personnes vivant effectivement dans le ménage
    2. le double du nombre de MII <- en partant du principe qu’au maximum deux personnes peuvent dormir sous une moustiquaire
   
* *Dénominateur* : le nombre de personnes ayant passé la nuit dans le ménage la veille de l’enquête

---

**L'utilisation des MII** se réfère au pourcentage des membres de fait du ménage ayant dormi la nuit précédant l'enquête sous une moustiquaire imprégnée d'insecticide (MII).

* *Numérateur* : Nombre de membres de fait du ménage ayant déclaré avoir dormi sous une moustiquaire imprégnée d'insecticide la nuit précédant l'enquête
* *Dénominateur* : Nombre de membres de fait du ménage

---

Pour plus d'informations (en anglais):
- Ressources relatives aux calculs des deux indicateurs
    - [Accès aux MII](https://dhsprogram.com/data/Guide-to-DHS-Statistics/Access_to_an_Insecticide-Treated_Net_ITN.htm)
    - [Utilisation des MII](https://dhsprogram.com/data/Guide-to-DHS-Statistics/index.htm#t=Use_of_Mosquito_Nets_by_Persons_in_the_Household.htm%23Percentage_of_the1bc-1&rhtocid=_15_3_0)
- [Les questionnaires utilisés dans les EDS/DHS](https://dhsprogram.com/publications/publication-dhsg4-dhs-questionnaires-and-manuals.cfm)

---

*Note* : Contrairement à la majorité des analyses dans le cadre du processus SNT, cette analyse est menée au niveau administratif **ADM1**, en raison de la disponibilité des données.

## 1. Configuration

In [ ]:
rm(list = ls())

options(scipen=999)

In [ ]:
# Global paths
Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.setenv(GDAL_DATA = "/opt/conda/share/gdal")

In [ ]:
# Paths
ROOT_PATH <- '~/workspace'
PIPELINE_PATH <- file.path(ROOT_PATH, 'pipelines', 'snt_dhs_indicators')
CONFIG_PATH <- file.path(ROOT_PATH, 'configuration')
CODE_PATH <- file.path(ROOT_PATH, 'code')
DATA_PATH <- file.path(ROOT_PATH, 'data')
DHS_DATA_PATH <- file.path(DATA_PATH, 'dhs', 'raw')
OUTPUT_DATA_PATH <- file.path(DATA_PATH, 'dhs', 'indicators', 'bednets')
OUTPUT_PLOTS_PATH <- file.path(ROOT_PATH, 'pipelines', 'snt_dhs_indicators', 'reporting', 'outputs')

In [ ]:
# Load notebook-specific utilities
source(file.path(CODE_PATH, "snt_utils.r"))
source(file.path(CODE_PATH, "snt_report.r"))
source(file.path(CODE_PATH, "snt_palettes.r"))
source(file.path(PIPELINE_PATH, "utils", "snt_dhs_bednets_report.r"))

# List required pcks
required_packages <- c("sf", "glue", "data.table", "ggplot2", "stringi", "jsonlite", "httr", "reticulate", "arrow", "IRdisplay")

# Execute function
install_and_load(required_packages)

In [ ]:
Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
reticulate::py_config()$python
openhexa <- import("openhexa.sdk")

In [ ]:
# Load SNT config
CONFIG_FILE_NAME <- "SNT_config.json"
config_json <- tryCatch({ fromJSON(file.path(CONFIG_PATH, CONFIG_FILE_NAME)) },
                        error = function(e) {
                          msg <- paste0("Error while loading configuration", conditionMessage(e))  
                          cat(msg)   
                          stop(msg) 
                        })

msg <- paste0("SNT configuration :", file.path(CONFIG_PATH, CONFIG_FILE_NAME)) 
log_msg(msg)

# Set config variables
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE

dhis2_dataset <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED

In [ ]:
admin_level <- 'ADM1'
admin_id_col <- glue(admin_level, 'ID', .sep='_')
admin_name_col <- glue(admin_level, 'NAME', .sep='_')
admin_cols <- c(admin_id_col, admin_name_col)

## 2. Chargement et pré-processing des données à visualiser

**Les données utilisées**

Toutes les données utilisées dans ce rapport sont agrégées au niveau administratif ADM1:

* Données spatiales : fond de carte (DHIS2)
* Données calculées par le pipeline:
   - valeurs estimées des indicateurs liés aux MII, ainsi que sur la précision statistique de ces estimations (intervalles de confiance à 95%) :
      - accès aux moustiquaires imprégnées d'insecticide
      - utilisation des moustiquaires imprégnées d'insecticide

In [ ]:
# Load spatial file from dataset
spatial_data_filename <- paste(COUNTRY_CODE, "shapes.geojson", sep = "_")
# spatial_data <- read_sf(file.path(DATA_PATH, 'dhis2', 'formatted', spatial_data_filename))
spatial_data <- get_latest_dataset_file_in_memory(dhis2_dataset, spatial_data_filename)
log_msg(glue("File {spatial_data_filename} successfully loaded from dataset version: {dhis2_dataset}"))

spatial_data <- st_as_sf(spatial_data)

# aggregate geometries by the admin columns
spatial_data <- aggregate_geometry(
  sf_data=spatial_data,
  admin_id_colname=admin_id_col,
  admin_name_colname=admin_name_col
)

# keep class
spatial_data <- st_as_sf(spatial_data)

if(COUNTRY_CODE == "COD"){
  spatial_data[[admin_name_col]] <- clean_admin_names(spatial_data[[admin_name_col]])
}

In [ ]:
# Import DHS data
data_source <- 'DHS'
indicator_access <- 'PCT_ITN_ACCESS'
indicator_use <- 'PCT_ITN_USE'

## 3. Création de graphiques


Pour chacun des deux indicateurs, le rapport crée deux visualisations:
   - Une carte choroplèthe indiquant l'estimation moyenne de la valeur de l'indicateur, sur base des données d'enquête
   - Un graphique de son intervalle de confiance, avec des barres d’erreur pour chaque ADM1

In [ ]:
filename_without_extension <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(indicator_access)}")
bednet_access_table <- fread(file.path(OUTPUT_DATA_PATH, paste0(filename_without_extension, '.csv')))

In [ ]:
access_plot_data =  merge(spatial_data, bednet_access_table, by = admin_cols, all = TRUE)

In [ ]:
access_lower_bound_col <- glue("{toupper(indicator_access)}_CI_LOWER_BOUND")
access_upper_bound_col <- glue("{toupper(indicator_access)}_CI_UPPER_BOUND")
access_sample_avg_col <- glue("{toupper(indicator_access)}_SAMPLE_AVERAGE")

### Accès aux MII

In [ ]:
access_plot <- make_pct_choropleth_map(
  map_data = access_plot_data,
  target_colname = access_sample_avg_col,
  plot_title = "Accès aux MII (%): Estimation moyenne",
  plot_subtitle = COUNTRY_CODE,
  plot_caption = glue("Données: {data_source}")
)

In [ ]:
access_plot_filename <- glue('{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(indicator_access)}_plot.png')
access_plot_path <- file.path(OUTPUT_PLOTS_PATH, access_plot_filename)
suppressMessages(ggsave(access_plot, file = access_plot_path, dpi = 500))

In [ ]:
display_png(file = access_plot_path)

In [ ]:
# confidence interval plot
access_ci_plot_title <- glue("Accès aux MII (Intervalles de Confiance 95%)")
access_ci_plot_ylab <- glue("Accès aux MII (%)")


In [ ]:
access_ci_plot <- make_ci_plot(
  df_to_plot=access_plot_data,
  admin_colname=admin_name_col,
  point_estimation_colname=access_sample_avg_col,
  ci_lower_colname=access_lower_bound_col,
  ci_upper_colname=access_upper_bound_col,
  plot_title=access_ci_plot_title,
  plot_subtitle=COUNTRY_CODE,
  plot_caption=glue("Données: {data_source}"),
  x_title=admin_level,
  y_title=access_ci_plot_ylab
)

In [ ]:
access_ci_plot_filename <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(indicator_access)}_CI_plot.png")
access_ci_plot_path <- file.path(OUTPUT_PLOTS_PATH, access_ci_plot_filename)
suppressMessages(ggsave(filename=access_ci_plot_path, plot=access_ci_plot, width = 6, height = 5, dpi = 300))

In [ ]:
display_png(file = access_ci_plot_path)

### Utilisation des MII

In [ ]:
filename_without_extension <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(indicator_use)}")
bednet_use_table <- fread(file.path(OUTPUT_DATA_PATH, paste0(filename_without_extension, '.csv')))

In [ ]:
use_plot_data =  merge(spatial_data, bednet_use_table, by = admin_cols, all = TRUE)

In [ ]:
use_lower_bound_col <- glue("{toupper(indicator_use)}_CI_LOWER_BOUND")
use_upper_bound_col <- glue("{toupper(indicator_use)}_CI_UPPER_BOUND")
use_sample_avg_col <- glue("{toupper(indicator_use)}_SAMPLE_AVERAGE")

In [ ]:
use_plot <- make_pct_choropleth_map(
  map_data = use_plot_data,
  target_colname = use_sample_avg_col,
  plot_title = "Utilisation des MII (%): Estimation moyenne",
  plot_subtitle = COUNTRY_CODE,
  plot_caption = glue("Données: {data_source}")
)

In [ ]:
use_plot_filename <- glue('{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(indicator_use)}_plot.png')
use_plot_path <- file.path(OUTPUT_PLOTS_PATH, use_plot_filename)
suppressMessages(ggsave(use_plot, file = use_plot_path, width = 6, height = 5, dpi = 300))

In [ ]:
display_png(file = use_plot_path)

In [ ]:
# confidence interval plot
use_ci_plot_title <- glue("Utilisation des MII (Intervalles de Confiance 95%)")
use_ci_plot_ylab <- glue("Utilisation des MII (%)")

In [ ]:
use_ci_plot <- make_ci_plot(
  df_to_plot=use_plot_data,
  admin_colname=admin_name_col,
  point_estimation_colname=use_sample_avg_col,
  ci_lower_colname=use_lower_bound_col,
  ci_upper_colname=use_upper_bound_col,
  plot_title=use_ci_plot_title,
  plot_subtitle=COUNTRY_CODE,
  plot_caption=glue("Données: {data_source}"),
  x_title=admin_level,
  y_title=use_ci_plot_ylab
)

In [ ]:
use_ci_plot_filename <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(indicator_use)}_CI_plot.png")
use_ci_plot_path <- file.path(OUTPUT_PLOTS_PATH, use_ci_plot_filename)
suppressMessages(ggsave(filename=use_ci_plot_path, plot=use_ci_plot, width = 6, height = 5, dpi = 300))

In [ ]:
display_png(file = use_ci_plot_path)